# 02 — PubMed Ingestion Pipeline

> **Stage:** Data Ingestion (RAG Knowledge Base)  
> **Source:** PubMed via BioPython (Entrez / Medline)

**Objective:** Build the ingestion pipeline to retrieve abstract-level medical literature. This data will eventually populate the Qdrant vector database for the RAG component.

Per the project's technical log (`DECISIONS.md`), we are adopting an **abstract-only retrieval strategy**. Abstracts are dense, structured, and highly available, fulfilling the clinical bibliographic assistance use case without the noise or access limitations of full-text articles.

---

**Sections**
1. Setup & Authentication
2. Exploring the Entrez API (MEDLINE format)
3. Building the Ingestion Function
4. Exporting Data

---

## 1. Setup & Authentication

In [1]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
from Bio import Entrez, Medline
from typing import List, Dict

Entrez.email = os.getenv("NCBI_EMAIL")
api_key = os.getenv("NCBI_API_KEY")
if api_key:
    Entrez.api_key = api_key

## 2. Exploring the Entrez API / MEDLINE Metadata
First do a manual test query to see what fields the MEDLINE format returns. This allows to map MEDLINE keys (like `TI`, `AB`, `MH`) to the target semantic schema.

In [3]:
# Fetching IDs for the query
query = "Colour perception deficiencies in children born prematurely"
max_results = 2

handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results, sort="relevance")
record = Entrez.read(handle)
handle.close()

record

{'Count': '2', 'RetMax': '2', 'RetStart': '0', 'IdList': ['37775921', '23841870'], 'TranslationSet': [{'From': 'Colour perception', 'To': '"colour perception"[All Fields] OR "color perception"[MeSH Terms] OR ("color"[All Fields] AND "perception"[All Fields]) OR "color perception"[All Fields]'}, {'From': 'deficiencies', 'To': '"deficiences"[All Fields] OR "deficiencies"[All Fields] OR "deficiency"[Subheading] OR "deficiency"[All Fields] OR "deficient"[All Fields] OR "deficients"[All Fields]'}, {'From': 'children', 'To': '"child"[MeSH Terms] OR "child"[All Fields] OR "children"[All Fields] OR "child\'s"[All Fields] OR "children\'s"[All Fields] OR "childrens"[All Fields] OR "childs"[All Fields]'}, {'From': 'born', 'To': '"parturition"[MeSH Terms] OR "parturition"[All Fields] OR "born"[All Fields]'}, {'From': 'prematurely', 'To': '"premature birth"[MeSH Terms] OR ("premature"[All Fields] AND "birth"[All Fields]) OR "premature birth"[All Fields] OR "premature"[All Fields] OR "prematurely"[A

In [4]:
# Download details from the retrieved IDs
handle = Entrez.efetch(db="pubmed", id=record["IdList"], rettype="medline", retmode="text")
records = list(Medline.parse(handle))
handle.close()

records

[{'PMID': '37775921',
  'OWN': 'NLM',
  'STAT': 'MEDLINE',
  'DCOM': '20240112',
  'LR': '20240815',
  'IS': '1651-2227 (Electronic) 0803-5253 (Linking)',
  'VI': '113',
  'IP': '2',
  'DP': '2024 Feb',
  'TI': 'Colour perception develops throughout childhood with increased risk of deficiencies in children born prematurely.',
  'PG': '259-266',
  'LID': '10.1111/apa.16978 [doi]',
  'AB': 'AIM: To quantify the impact of prematurity on chromatic discrimination throughout childhood, from 2 to 15 years of age. METHODS: We recruited two cohorts of children, as part of the TrackAI Project, an international project with seven different study sites: a control group of full-term children with normal visual development and a group of children born prematurely. All children underwent a complete ophthalmological exam and an assessment of colour discrimination along the three colour axes: deutan, protan and trytan using a DIVE device with eye tracking technology. RESULTS: We enrolled a total of 187

In [5]:
# Extract relevant fields
final_fields = ["PMID", "TI", "AB", "AU", "OT", "MH", "JT", "DP", "LID"]
row = {
    "title": records[0].get("TI", ""),
    "abstract": records[0].get("AB", ""),
    "mesh": records[0].get("MH", ""),
    "authors": "; ".join(records[0].get("AU", [])),
    "author_keywords": records[0].get("OT", ""),
    "journal": records[0].get("JT", ""),
    "year": records[0].get("DP", "")[:4],
    "pmid": records[0].get("PMID", ""),
    "doi": records[0].get("LID", "").split()[0] if "LID" in records[0] else "",
    "publication_types": records[0].get("PT", []),
}

row

{'title': 'Colour perception develops throughout childhood with increased risk of deficiencies in children born prematurely.',
 'abstract': 'AIM: To quantify the impact of prematurity on chromatic discrimination throughout childhood, from 2 to 15 years of age. METHODS: We recruited two cohorts of children, as part of the TrackAI Project, an international project with seven different study sites: a control group of full-term children with normal visual development and a group of children born prematurely. All children underwent a complete ophthalmological exam and an assessment of colour discrimination along the three colour axes: deutan, protan and trytan using a DIVE device with eye tracking technology. RESULTS: We enrolled a total of 1872 children (928 females and 944 males) with a mean age of 6.64 years. Out of them, 374 were children born prematurely and 1498 were full-term controls. Using data from all the children born at term, reference normative curves were plotted for colour d

## 3. Building the Ingestion Function
Wrap the search and fetch logic into a reusable function that parses the text output, filters out records without abstracts, and maps the proprietary MEDLINE keys into a clean dictionary.

In [6]:
import time
from http.client import IncompleteRead
from urllib.error import HTTPError, URLError

EFETCH_BATCH_SIZE = 200       # NCBI guidance: keep efetch batches <= ~500
MAX_RETRIES = 4
RETRY_BACKOFF = 2.0           # seconds; doubled each retry


def _efetch_batch(webenv: str, query_key: str, retstart: int, retmax: int):
    """Single efetch call against the history server, with retries."""
    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            handle = Entrez.efetch(
                db="pubmed",
                rettype="medline",
                retmode="text",
                webenv=webenv,
                query_key=query_key,
                retstart=retstart,
                retmax=retmax,
            )
            records = list(Medline.parse(handle))
            handle.close()
            return records
        except (IncompleteRead, HTTPError, URLError, ConnectionError) as e:
            last_err = e
            wait = RETRY_BACKOFF * (2 ** attempt)
            print(f"    efetch retstart={retstart} failed ({type(e).__name__}); "
                  f"retry {attempt + 1}/{MAX_RETRIES} in {wait:.1f}s")
            time.sleep(wait)
    raise RuntimeError(f"efetch failed after {MAX_RETRIES} retries: {last_err}")


def fetch_pubmed_abstracts(
    query: str, max_results: int = 15
) -> List[Dict[str, str]]:
    """Search PubMed and return abstracts with metadata.

    Uses the NCBI history server (usehistory=y) and batched efetch so large
    queries (thousands of IDs) don't fail with IncompleteRead.
    """
    # 1. esearch with history → get WebEnv + QueryKey instead of an ID list.
    handle = Entrez.esearch(
        db="pubmed",
        term=query,
        retmax=max_results,
        sort="relevance",
        usehistory="y",
    )
    record = Entrez.read(handle)
    handle.close()

    count = min(int(record["Count"]), max_results)
    print(f"Found {count} papers for query: '{query[:120]}{'...' if len(query) > 120 else ''}'")
    if count == 0:
        return []

    webenv = record["WebEnv"]
    query_key = record["QueryKey"]

    # 2. Paginated efetch via the history server.
    all_records = []
    for retstart in range(0, count, EFETCH_BATCH_SIZE):
        batch = _efetch_batch(
            webenv=webenv,
            query_key=query_key,
            retstart=retstart,
            retmax=min(EFETCH_BATCH_SIZE, count - retstart),
        )
        print(f"  Downloaded records {retstart + 1}–{retstart + len(batch)}", end="")
        all_records.extend(batch)
        time.sleep(0.15)  # polite pacing

    # 3. Map MEDLINE keys → clean schema; drop records without abstracts.
    papers = []
    for r in all_records:
        abstract = r.get("AB", "")
        if abstract:
            papers.append({
                "title": r.get("TI", ""),
                "abstract": abstract,
                "mesh": r.get("MH", ""),
                "authors": "; ".join(r.get("AU", [])),
                "author_keywords": r.get("OT", ""),
                "journal": r.get("JT", ""),
                "year": r.get("DP", "")[:4],
                "pmid": r.get("PMID", ""),
                "doi": r.get("LID", "").split()[0] if "LID" in r else "",
                "publication_types": r.get("PT", []),
            })

    print(f"Downloaded {len(papers)} papers with abstracts")
    return papers


In [7]:
# Example usage

# Download a sample set of papers
papers = fetch_pubmed_abstracts("Colour perception deficiencies in children born prematurely", max_results=15)

# Quick preview
for p in papers[:3]:
    print(f"\n[{p['pmid']}]-[{p['doi']}] {p['title'][:80]}...")
    print(f"   {p['abstract'][:150]}...")

Found 2 papers for query: 'Colour perception deficiencies in children born prematurely'
  Downloaded records 1–2Downloaded 2 papers with abstracts

[37775921]-[10.1111/apa.16978] Colour perception develops throughout childhood with increased risk of deficienc...
   AIM: To quantify the impact of prematurity on chromatic discrimination throughout childhood, from 2 to 15 years of age. METHODS: We recruited two coho...

[23841870]-[10.1111/dmcn.12206] Compromised approximate number system acuity in extremely preterm school-aged ch...
   AIM: The aim of this study was to compare the approximate number system acuity in children born extremely preterm aged 6 years 6 months and typically ...


## 4. Exporting Data
    
The final and most important step for this notebook is to **export the scraped data to disk**. 

The goal is to avoid pinging the NCBI API repeatedly in downstream pipelines. The embeddings generated in the next phase (`03_embedding_benchmark.ipynb`) will load this local JSON file instead of making live web requests.

In [8]:
import json
import os

# Ensure the output directory exists
output_dir = "../data/raw"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "pubmed_sample.json")

# Save the dataset to disk
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(papers, f, indent=2, ensure_ascii=False)

print(f"Successfully saved {len(papers)} papers to {output_path}")

Successfully saved 2 papers to ../data/raw/pubmed_sample.json


## 5. Bulk Ingestion (test functionality, not final ingestion)

To train our model and populate the RAG vector database, we need a broad corpus of medical literature. 
Instead of querying PubMed for every single MTSamples record (~5,000 queries), we will extract the unique **40 medical specialties** and fetch 50-100 high-relevance papers for each. This builds a robust, clustered knowledge base of ~4,000 abstracts.

In [ ]:
import re
import time

import pandas as pd

# 1. Load specialties from MTSamples
MTSAMPLES_PATH = "hf://datasets/harishnair04/mtsamples/mtsamples.csv"
df = pd.read_csv(MTSAMPLES_PATH)

# Clean and extract unique specialties
specialties = df["medical_specialty"].dropna().unique()
specialties = [s.strip() for s in specialties if str(s).strip()]
print(f"Found {len(specialties)} unique specialties.\n")

# 2. Fetch papers for each specialty
all_specialty_papers = {}
papers_per_specialty = 50

# NOTE: Sliced to [:3] for safe testing. Remove the slice to run the full bulk download
for spec in specialties:
    print(f"=== Fetching for: {spec} ===")

    # You can enrich the query if needed, e.g., f"{spec} medical guidelines"
    query = spec

    papers = fetch_pubmed_abstracts(query=query, max_results=papers_per_specialty)
    all_specialty_papers[spec] = papers

    # Be respectful to the NCBI API limits (max 3 req/sec without key, 10 with key)
    time.sleep(1)

# 3. Export the bulk dataset
corpus_path = os.path.join(output_dir, "pubmed_bulk_corpus.json")
with open(corpus_path, "w", encoding="utf-8") as f:
    json.dump(all_specialty_papers, f, indent=2, ensure_ascii=False)

total_papers = sum(len(p) for p in all_specialty_papers.values())
print(f"\nBulk ingestion complete. Saved {total_papers} papers across {len(all_specialty_papers)} specialties to {corpus_path}")

Found 40 unique specialties.

=== Fetching for: Allergy / Immunology ===
Found 50 papers for query: 'Allergy / Immunology'
Downloaded 20 papers with abstracts
=== Fetching for: Bariatrics ===
Found 50 papers for query: 'Bariatrics'
Downloaded 28 papers with abstracts
=== Fetching for: Cardiovascular / Pulmonary ===
Found 50 papers for query: 'Cardiovascular / Pulmonary'
Downloaded 35 papers with abstracts
=== Fetching for: Neurology ===
Found 50 papers for query: 'Neurology'
Downloaded 21 papers with abstracts
=== Fetching for: Dentistry ===
Found 50 papers for query: 'Dentistry'
Downloaded 25 papers with abstracts
=== Fetching for: Urology ===
Found 50 papers for query: 'Urology'
Downloaded 25 papers with abstracts
=== Fetching for: General Medicine ===
Found 50 papers for query: 'General Medicine'
Downloaded 35 papers with abstracts
=== Fetching for: Surgery ===
Found 50 papers for query: 'Surgery'
Downloaded 12 papers with abstracts
=== Fetching for: Speech - Language ===
Found 50 p

## 6. Broad MeSH-anchored Ingestion

The standard approach used in clinical RAG systems (Almanac, Clinfo.ai,
MedRAG benchmark) is to **decouple corpus from queries**:

1. Build a broad reference corpus once, anchored on the **MeSH disease
   tree** top-level descriptors (Tree C + F03 mental disorders + key E
   procedure trees). Each query uses `[MeSH Major Topic]` so we only get
   papers where the descriptor is the central focus, not incidentally
   indexed.
2. Apply a publication-type whitelist (Reviews / Guidelines / RCTs /
   Meta-Analyses / Clinical Trials / Systematic Reviews) and a recency
   filter (≥ 2015).
3. Tag each abstract with its `mesh_category` so downstream code can
   filter or stratify if needed.
4. At query time, BioBERT + Qdrant dense retrieval handles relevance —
   the whole point of dense retrieval is that you don't need to
   pre-curate the corpus per query.

Result: a flat `data/raw/pubmed_bulk_corpus.json` of ~30–50K high-quality
abstracts, each tagged with its MeSH category. Replaces all earlier
versions of this file.

See `DECISIONS.md` (2026-04-28 entry "Broad MeSH-anchored PubMed corpus")
for full rationale.


In [9]:
"""Broad MeSH-anchored PubMed corpus (v3 — final).

Drives ingestion off the MeSH disease tree (Tree C + F03 + selected E trees)
using `[MeSH Major Topic]`, with publication-type whitelisting and a recency
filter. See DECISIONS.md (2026-04-28 entry "Broad MeSH-anchored PubMed corpus").

Output: flat list of papers, each tagged with `mesh_category`. The retriever
(BioBERT + Qdrant) handles per-query relevance at retrieval time.
"""
import time
import json
from collections import Counter

output_dir = "../data/raw"

# Top-level MeSH descriptors covering the diseases / procedures relevant to
# MTSamples. Tree C (diseases), F03 (mental disorders), E01 (diagnostics),
# E02-E04 (therapeutics / surgery).
MESH_CATEGORIES = [
    # Diseases (C tree, top-level)
    "Bacterial Infections and Mycoses",
    "Virus Diseases",
    "Parasitic Diseases",
    "Neoplasms",
    "Musculoskeletal Diseases",
    "Digestive System Diseases",
    "Stomatognathic Diseases",
    "Respiratory Tract Diseases",
    "Otorhinolaryngologic Diseases",
    "Nervous System Diseases",
    "Eye Diseases",
    "Urogenital Diseases",
    "Cardiovascular Diseases",
    "Hemic and Lymphatic Diseases",
    "Congenital, Hereditary, and Neonatal Diseases and Abnormalities",
    "Skin and Connective Tissue Diseases",
    "Nutritional and Metabolic Diseases",
    "Endocrine System Diseases",
    "Immune System Diseases",
    "Wounds and Injuries",
    # Mental disorders
    "Mental Disorders",
    # Procedures aligned with MTSamples surgical/diagnostic specialties
    "Surgical Procedures, Operative",
    "Diagnostic Techniques and Procedures",
    "Therapeutics",
]

PAPERS_PER_CATEGORY = 2000  # ~50K total upper bound (after dedup)
RECENCY_FROM_YEAR = 2015

PT_WHITELIST = (
    'Review[PT] OR "Practice Guideline"[PT] OR Guideline[PT] OR '
    '"Randomized Controlled Trial"[PT] OR "Meta-Analysis"[PT] OR '
    '"Clinical Trial"[PT] OR "Systematic Review"[PT]'
)
PT_BLACKLIST = (
    'Editorial[PT] OR Letter[PT] OR Comment[PT] OR News[PT] OR '
    'Biography[PT] OR "Personal Narrative"[PT]'
)


def build_mesh_query(descriptor: str) -> str:
    return (
        f'"{descriptor}"[MeSH Major Topic] '
        f'AND ({RECENCY_FROM_YEAR}:3000[dp]) '
        f'AND ({PT_WHITELIST}) '
        f'NOT ({PT_BLACKLIST})'
    )


# 1. Fetch per MeSH category, dedup by PMID across categories.
corpus: list[dict] = []
seen_pmids: set[str] = set()
per_category_counts: dict[str, tuple[int, int]] = {}

for cat in MESH_CATEGORIES:
    q = build_mesh_query(cat)
    print(f"\n=== {cat} ===")
    papers = fetch_pubmed_abstracts(q, max_results=PAPERS_PER_CATEGORY)
    new_count = 0
    for p in papers:
        pmid = p.get("pmid")
        if pmid and pmid not in seen_pmids:
            seen_pmids.add(pmid)
            corpus.append({**p, "mesh_category": cat})
            new_count += 1
    per_category_counts[cat] = (len(papers), new_count)
    print(f"  fetched={len(papers)}  new={new_count}  total_unique={len(corpus)}")
    time.sleep(0.15)  # ~7 req/s, under the 10 req/s API-key limit

# 2. Save flat corpus.
out_path = os.path.join(output_dir, "pubmed_bulk_corpus.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(corpus, f, indent=2, ensure_ascii=False)

# 3. Summary.
cat_counter = Counter(p["mesh_category"] for p in corpus)
print("\n=== Bulk MeSH-anchored corpus ===")
print(f"Categories queried:     {len(MESH_CATEGORIES)}")
print(f"Total unique abstracts: {len(corpus)}")
print(f"Output: {out_path}")
print("\nDistribution by MeSH category:")
for cat, n in cat_counter.most_common():
    print(f"  {n:5d}  {cat}")



=== Bacterial Infections and Mycoses ===
Found 2000 papers for query: '"Bacterial Infections and Mycoses"[MeSH Major Topic] AND (2015:3000[dp]) AND (Review[PT] OR "Practice Guideline"[PT] OR ...'
  Downloaded records 1–200  Downloaded records 201–400  Downloaded records 401–600  Downloaded records 601–800  Downloaded records 801–1000  Downloaded records 1001–1200  Downloaded records 1201–1400  Downloaded records 1401–1600  Downloaded records 1601–1800  Downloaded records 1801–2000Downloaded 1924 papers with abstracts
  fetched=1924  new=1924  total_unique=1924

=== Virus Diseases ===
Found 2000 papers for query: '"Virus Diseases"[MeSH Major Topic] AND (2015:3000[dp]) AND (Review[PT] OR "Practice Guideline"[PT] OR Guideline[PT] OR "...'
  Downloaded records 1–200  Downloaded records 201–400    efetch retstart=400 failed (IncompleteRead); retry 1/4 in 2.0s
  Downloaded records 401–600  Downloaded records 601–800  Downloaded records 801–1000  Downloaded records 1001–1200  Downloaded reco